In [1]:

# ==============================
# Standard Libraries
# ==============================

import warnings
warnings.filterwarnings("ignore")

import os
import re
import string

from pathlib import Path

# ==============================
# Data Handling
# ==============================

import numpy as np
import pandas as pd

# ==============================
# Visualization
# ==============================

import matplotlib.pyplot as plt

# ==============================
# Machine Learning
# ==============================

from sklearn.model_selection import train_test_split

# ==============================
# NLP
# ==============================

import nltk
import spacy

from nltk.corpus import stopwords

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
nltk.download("stopwords")
nltk.download("punkt")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))

print("spaCy Loaded Successfully")

spaCy Loaded Successfully


In [4]:
BASE_DIR = Path.cwd()

DATASET_DIR = BASE_DIR / "dataset"

In [5]:
import os

print(os.getcwd())

C:\Users\Lenovo\Resume_AI_Research\notebooks


In [6]:
import os

print(os.listdir())

['.ipynb_checkpoints', '01_Data_Loading.ipynb.ipynb', '02_Data_Preprocessing.ipynb.ipynb', '03_Feature_Engineering.ipynb.ipynb', '04_NLP_Preprocessing.ipynb.ipynb', 'best_resume_model.pkl', 'Research_Implementation.ipynb', 'scaler.pkl', 'tfidf_vectorizer.pkl']


In [7]:
from pathlib import Path

print("Current Directory:", Path.cwd())
print("Parent Directory :", Path.cwd().parent)

Current Directory: C:\Users\Lenovo\Resume_AI_Research\notebooks
Parent Directory : C:\Users\Lenovo\Resume_AI_Research


In [8]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATASET_DIR = BASE_DIR / "dataset"

print(DATASET_DIR)

C:\Users\Lenovo\Resume_AI_Research\dataset


In [9]:
import os

print(os.listdir(DATASET_DIR))

['.ipynb_checkpoints', 'job_descriptions', 'processed', 'resumes', 'resume_data.csv', 'resume_data_for_ranking.csv']


In [10]:
resume_df = pd.read_csv(DATASET_DIR / "Resume_Data_For_Ranking.csv")

In [11]:
resume_df.shape

(9544, 35)

In [12]:
print(os.listdir(DATASET_DIR))

['.ipynb_checkpoints', 'job_descriptions', 'processed', 'resumes', 'resume_data.csv', 'resume_data_for_ranking.csv']


In [13]:
resume_df.columns = (
    resume_df.columns
    .str.strip()
    .str.lower()
)

resume_df.columns.tolist()

['address',
 'career_objective',
 'skills',
 'educational_institution_name',
 'degree_names',
 'passing_years',
 'educational_results',
 'result_types',
 'major_field_of_studies',
 'professional_company_names',
 'company_urls',
 'start_dates',
 'end_dates',
 'related_skils_in_job',
 'positions',
 'locations',
 'responsibilities',
 'extra_curricular_activity_types',
 'extra_curricular_organization_names',
 'extra_curricular_organization_links',
 'role_positions',
 'languages',
 'proficiency_levels',
 'certification_providers',
 'certification_skills',
 'online_links',
 'issue_dates',
 'expiry_dates',
 'job_position_name',
 'educational_requirements',
 'experiencere_requirement',
 'age_requirement',
 'responsibilities.1',
 'skills_required',
 'matched_score']

In [14]:
candidate_columns = [
    "career_objective",
    "skills",
    "degree_names",
    "major_field_of_studies",
    "professional_company_names",
    "positions",
    "certification_skills"
]

job_columns = [
    "job_position_name",
    "skills_required",
    "educational_requirements",
    "experiencere_requirement"
]

In [15]:
required_columns = candidate_columns + job_columns

missing = [c for c in required_columns if c not in resume_df.columns]

print(missing)

[]


In [16]:
resume_df[required_columns] = (
    resume_df[required_columns]
    .fillna("")
)

In [17]:
resume_df["candidate_profile"] = (
    resume_df[candidate_columns]
    .astype(str)
    .agg(" ".join, axis=1)
)

In [18]:
resume_df["job_profile"] = (
    resume_df[job_columns]
    .astype(str)
    .agg(" ".join, axis=1)
)

In [19]:
resume_df[
    [
        "candidate_profile",
        "job_profile",
        "matched_score"
    ]
].head()

,candidate_profile,job_profile,matched_score
0,Big data analytics working and database wareho...,Senior Software Engineer B.Sc in Computer Sci...,0.850000
1,Fresher looking to join as a data analyst and ...,Machine Learning (ML) Engineer M.Sc in Comput...,0.750000
2,"['Software Development', 'Machine Learning', ...","Executive/ Senior Executive- Trade Marketing, ...",0.416667
3,To obtain a position in a fast-paced business ...,Business Development Executive Fast typing ski...,0.760000
4,Professional accountant with an outstanding wo...,Senior iOS Engineer iOS\niOS App Developer\niO...,0.650000


In [20]:
PROCESSED_DIR = DATASET_DIR / "processed"

PROCESSED_DIR.mkdir(exist_ok=True)

resume_df.to_csv(
    PROCESSED_DIR / "profiles_created.csv",
    index=False
)

print("Profiles saved successfully.")

Profiles saved successfully.


In [21]:
import re
import string
import pandas as pd
import spacy
from nltk.corpus import stopwords

In [22]:
nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))

print("spaCy Loaded")

spaCy Loaded


In [23]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+", " ", text)

    # remove emails
    text = re.sub(r"\S+@\S+", " ", text)

    # remove digits
    text = re.sub(r"\d+", " ", text)

    # remove punctuation
    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [24]:
def preprocess_text(text):

    text = clean_text(text)

    doc = nlp(text)

    tokens = []

    for token in doc:

        if token.is_stop:
            continue

        if token.is_punct:
            continue

        if token.lemma_ == "-PRON-":
            continue

        tokens.append(token.lemma_)

    return " ".join(tokens)

In [25]:
sample = resume_df["candidate_profile"].iloc[0]

print(sample)

Big data analytics working and database warehouse manager with robust experience in handling all kinds of data. I have also used multiple cloud infrastructure services and am well acquainted with them. Currently in search of role that offers more of development. ['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapreduce', 'Spark', 'Java', 'Machine Learning', 'Cloud', 'Hdfs', 'YARN', 'Core Java', 'Data Science', 'C++', 'Data Structures', 'DBMS', 'RDBMS', 'Informatica', 'Talend', 'Amazon Redshift', 'Microsoft Azure'] ['B.Tech'] ['Electronics'] ['Coca-COla'] ['Big Data Analyst'] 


In [26]:
cleaned = preprocess_text(sample)

print(cleaned)

big data analytic work database warehouse manager robust experience handle kind datum multiple cloud infrastructure service acquainted currently search role offer development big data hadoop hive python mapreduce spark java machine learning cloud hdfs yarn core java datum science c data structure dbms rdbms informatica talend amazon redshift microsoft azure btech electronic cocacola big data analyst


In [ ]:
resume_df["candidate_clean"] = (
    resume_df["candidate_profile"]
    .apply(preprocess_text)
)

resume_df["job_clean"] = (
    resume_df["job_profile"]
    .apply(preprocess_text)
)

In [ ]:
resume_df[
    [
        "candidate_clean",
        "job_clean",
        "matched_score"
    ]
].head()

In [ ]:
resume_df.to_csv(
    DATASET_DIR / "processed" / "cleaned_profiles.csv",
    index=False
)

print("✅ Cleaned dataset saved successfully.")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words="english"
)

In [ ]:
combined_text = pd.concat([
    resume_df["candidate_clean"],
    resume_df["job_clean"]
])

tfidf.fit(combined_text)

print("Vocabulary Size:", len(tfidf.vocabulary_))

In [ ]:
candidate_vectors = tfidf.transform(
    resume_df["candidate_clean"]
)

In [ ]:
job_vectors = tfidf.transform(
    resume_df["job_clean"]
)

In [ ]:
tfidf_similarity = []

for i in range(len(resume_df)):
    
    score = cosine_similarity(
        candidate_vectors[i],
        job_vectors[i]
    )[0][0]
    
    tfidf_similarity.append(score)

resume_df["tfidf_similarity"] = tfidf_similarity

In [ ]:
resume_df[
    [
        "tfidf_similarity",
        "matched_score"
    ]
].head()

In [ ]:
resume_df["tfidf_similarity"].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    resume_df["tfidf_similarity"],
    bins=30
)

plt.title("Distribution of TF-IDF Similarity")
plt.xlabel("Similarity Score")
plt.ylabel("Frequency")

plt.show()

In [ ]:
resume_df.to_csv(
    DATASET_DIR / "processed" / "tfidf_features.csv",
    index=False
)

print("✅ TF-IDF features saved successfully.")

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence-BERT Loaded")

In [ ]:
candidate_embeddings = model.encode(
    resume_df["candidate_clean"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

In [ ]:
print("candidate_embeddings" in globals())
print("job_embeddings" in globals())

In [ ]:
print(candidate_embeddings.shape)

In [ ]:
job_embeddings = model.encode(
    resume_df["job_clean"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

In [ ]:
print(job_embeddings.shape)

In [ ]:
from sklearn.metrics.pairwise import paired_cosine_distances

resume_df["semantic_similarity"] = (
    1 - paired_cosine_distances(
        candidate_embeddings,
        job_embeddings
    )
)

resume_df["semantic_similarity"].head()

In [ ]:
resume_df["semantic_similarity"].describe()

In [ ]:
semantic_similarity = []

for i in range(len(resume_df)):
    score = cosine_similarity(
        candidate_embeddings[i].reshape(1, -1),
        job_embeddings[i].reshape(1, -1)
    )[0][0]

    semantic_similarity.append(score)

resume_df["semantic_similarity"] = semantic_similarity

In [ ]:
candidate_embeddings.shape

In [ ]:
job_embeddings = model.encode(
    resume_df["job_clean"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

In [ ]:
job_embeddings.shape

In [ ]:
from sklearn.metrics.pairwise import paired_cosine_distances

resume_df["semantic_similarity"] = (
    1 - paired_cosine_distances(
        candidate_embeddings,
        job_embeddings
    )
)

In [ ]:
from sklearn.metrics.pairwise import paired_cosine_distances

resume_df["semantic_similarity"] = (
    1 - paired_cosine_distances(
        candidate_embeddings,
        job_embeddings
    )
)

In [ ]:
resume_df["semantic_similarity"].head()

In [ ]:
candidate_embeddings.shape

In [ ]:
job_embeddings.shape

In [ ]:
def extract_skills(skill_text):

    if pd.isna(skill_text):
        return set()

    skill_text = str(skill_text).lower()

    skill_text = skill_text.replace("[", "")
    skill_text = skill_text.replace("]", "")
    skill_text = skill_text.replace("'", "")

    skills = [
        s.strip()
        for s in skill_text.split(",")
        if s.strip()
    ]

    return set(skills)

In [ ]:
skill_scores = []

for _, row in resume_df.iterrows():

    candidate = extract_skills(row["skills"])

    required = extract_skills(row["skills_required"])

    if len(required) == 0:
        score = 0

    else:
        score = len(candidate & required) / len(required)

    skill_scores.append(score)

resume_df["skill_match_score"] = skill_scores

In [ ]:
resume_df[
    [
        "skills",
        "skills_required",
        "skill_match_score"
    ]
].head()

In [ ]:
education_rank = {
    "high school": 1,
    "diploma": 2,
    "b.sc": 3,
    "b.tech": 3,
    "b.e": 3,
    "bachelor": 3,
    "m.sc": 4,
    "m.tech": 4,
    "mba": 4,
    "master": 4,
    "phd": 5,
    "doctorate": 5
}

In [ ]:
def education_score(candidate, required):

    candidate = str(candidate).lower()
    required = str(required).lower()

    candidate_level = 0
    required_level = 0

    for degree, level in education_rank.items():

        if degree in candidate:
            candidate_level = max(candidate_level, level)

        if degree in required:
            required_level = max(required_level, level)

    if required_level == 0:
        return 0.5

    if candidate_level >= required_level:
        return 1.0

    return candidate_level / required_level

In [ ]:
resume_df["education_match_score"] = resume_df.apply(
    lambda row: education_score(
        row["degree_names"],
        row["educational_requirements"]
    ),
    axis=1
)

In [ ]:
resume_df[
    [
        "degree_names",
        "educational_requirements",
        "education_match_score"
    ]
].head()

In [ ]:
import re

def extract_years(text):

    text = str(text)

    numbers = re.findall(r"\d+", text)

    if len(numbers) == 0:
        return 0

    return int(numbers[0])

In [ ]:
def experience_score(candidate, required):

    c = extract_years(candidate)
    r = extract_years(required)

    if r == 0:
        return 0.5

    if c >= r:
        return 1.0

    return c / r

In [ ]:
resume_df["experience_match_score"] = resume_df.apply(
    lambda row: experience_score(
        row["positions"],
        row["experiencere_requirement"]
    ),
    axis=1
)

In [ ]:
resume_df[
    [
        "positions",
        "experiencere_requirement",
        "experience_match_score"
    ]
].head()

In [ ]:
def certification_score(candidate, required):

    candidate = set(
        str(candidate).lower().split(",")
    )

    required = set(
        str(required).lower().split(",")
    )

    candidate = {x.strip() for x in candidate if x.strip()}
    required = {x.strip() for x in required if x.strip()}

    if len(required) == 0:
        return 0.5

    return len(candidate & required) / len(required)

In [ ]:
resume_df["certification_match_score"] = resume_df.apply(
    lambda row: certification_score(
        row["certification_skills"],
        row["skills_required"]
    ),
    axis=1
)

In [ ]:
resume_df[
    [
        "certification_skills",
        "skills_required",
        "certification_match_score"
    ]
].head()

In [ ]:
features = [
    "tfidf_similarity",
    "semantic_similarity",
    "skill_match_score",
    "education_match_score",
    "experience_match_score",
    "certification_match_score"
]

X = resume_df[features]

y = resume_df["matched_score"]

print("Feature Matrix Shape :", X.shape)
print("Target Shape :", y.shape)

In [ ]:
X.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train)

print("Model Trained Successfully")

In [ ]:
y_pred = lr.predict(X_test)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse**0.5

r2 = r2_score(
    y_test,
    y_pred
)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(
    X_train,
    y_train
)

In [ ]:
rf_pred = rf.predict(
    X_test
)

In [ ]:
rf_mae = mean_absolute_error(
    y_test,
    rf_pred
)

rf_rmse = mean_squared_error(
    y_test,
    rf_pred
) ** 0.5

rf_r2 = r2_score(
    y_test,
    rf_pred
)

print("Random Forest")

print("MAE :", rf_mae)

print("RMSE:", rf_rmse)

print("R² :", rf_r2)

In [ ]:
comparison = pd.DataFrame({

    "Model":[

        "Linear Regression",

        "Random Forest"

    ],

    "MAE":[

        mae,

        rf_mae

    ],

    "RMSE":[

        rmse,

        rf_rmse

    ],

    "R2":[

        r2,

        rf_r2

    ]

})

comparison

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
!pip install shap

In [ ]:
import shap
import matplotlib.pyplot as plt

In [ ]:
explainer = shap.Explainer(lr, X_train)

In [ ]:
shap_values = explainer(X_test)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
shap.plots.bar(shap_values)

In [ ]:
shap.plots.waterfall(shap_values[0])

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd().parent   # because notebook is inside notebooks/

FIGURES_DIR = BASE_DIR / "figures"

FIGURES_DIR.mkdir(exist_ok=True)

print(FIGURES_DIR)

In [ ]:
plt.savefig(
    FIGURES_DIR / "shap_summary.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
sample_index = 0

shap.plots.waterfall(
    shap_values[sample_index]
)

In [ ]:
plt.savefig(
    DATASET_DIR / "processed" / "shap_summary.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
resume_df["predicted_score"] = lr.predict(X)

In [ ]:
ranking = resume_df[
    [
        "job_position_name",
        "predicted_score"
    ]
].sort_values(
    by="predicted_score",
    ascending=False
)

ranking.head(10)

In [ ]:
ranking.to_csv(
    DATASET_DIR / "processed" / "candidate_ranking.csv",
    index=False
)

print("Ranking saved successfully.")

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr.coef_
})

importance = importance.sort_values(
    by="Coefficient",
    key=abs,
    ascending=False
)

importance

In [ ]:
plt.figure(figsize=(8,5))

plt.barh(
    importance["Feature"],
    importance["Coefficient"]
)

plt.xlabel("Coefficient")

plt.title("Linear Regression Feature Importance")

plt.show()

In [ ]:
resume_df["resume_length"] = (
    resume_df["candidate_clean"]
    .str.len()
)

In [ ]:
resume_df["job_length"] = (
    resume_df["job_clean"]
    .str.len()
)

In [ ]:
resume_df["resume_word_count"] = (
    resume_df["candidate_clean"]
    .str.split()
    .str.len()
)

In [ ]:
resume_df["job_word_count"] = (
    resume_df["job_clean"]
    .str.split()
    .str.len()
)

In [ ]:
resume_df["candidate_skill_count"] = (
    resume_df["skills"]
    .fillna("")
    .apply(lambda x: len(extract_skills(x)))
)

In [ ]:
resume_df["required_skill_count"] = (
    resume_df["skills_required"]
    .fillna("")
    .apply(lambda x: len(extract_skills(x)))
)

In [ ]:
def skill_overlap(candidate, required):
    candidate = extract_skills(candidate)
    required = extract_skills(required)
    return len(candidate & required)

resume_df["skill_overlap_count"] = resume_df.apply(
    lambda row: skill_overlap(
        row["skills"],
        row["skills_required"]
    ),
    axis=1
)

In [ ]:
resume_df["education_exact_match"] = resume_df.apply(
    lambda row: int(
        str(row["degree_names"]).lower()
        in str(row["educational_requirements"]).lower()
    ),
    axis=1
)

In [ ]:
new_features = [
    "resume_length",
    "job_length",
    "resume_word_count",
    "job_word_count",
    "candidate_skill_count",
    "required_skill_count",
    "skill_overlap_count",
    "education_exact_match"
]

resume_df[new_features].head()

In [ ]:
features = [
    "tfidf_similarity",
    "semantic_similarity",
    "skill_match_score",
    "education_match_score",
    "experience_match_score",
    "certification_match_score",
    "resume_length",
    "job_length",
    "resume_word_count",
    "job_word_count",
    "candidate_skill_count",
    "required_skill_count",
    "skill_overlap_count",
    "education_exact_match"
]

X = resume_df[features]
y = resume_df["matched_score"]

print(X.shape)

In [ ]:
X.shape

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.001),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)

    rmse = mean_squared_error(
        y_test,
        pred
    ) ** 0.5

    r2 = r2_score(y_test, pred)

    results.append([
        name,
        mae,
        rmse,
        r2
    ])

In [ ]:
comparison = pd.DataFrame(
    results,
    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R2"
    ]
)

comparison = comparison.sort_values(
    by="R2",
    ascending=False
)

comparison

In [ ]:
plt.figure(figsize=(10,5))

plt.bar(
    comparison["Model"],
    comparison["R2"]
)

plt.xticks(rotation=20)

plt.ylabel("R² Score")

plt.title("Comparison of Regression Models")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.bar(
    comparison["Model"],
    comparison["RMSE"]
)

plt.xticks(rotation=20)

plt.ylabel("RMSE")

plt.title("RMSE Comparison")

plt.show()

In [ ]:
best_model_name = comparison.iloc[0]["Model"]

print("Best Model:", best_model_name)

In [ ]:
best_model = models[best_model_name]

best_model.fit(X_train, y_train)

In [ ]:
print(best_model_name)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

In [ ]:
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

In [ ]:
print("Best Parameters:")
print(grid_search.best_params_)

In [ ]:
best_rf = grid_search.best_estimator_

In [ ]:
pred = best_rf.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE :", mean_absolute_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)
print("R²  :", r2_score(y_test, pred))

In [ ]:
import shap

explainer = shap.TreeExplainer(best_rf)

shap_values = explainer.shap_values(X_test)

In [ ]:
feature_names = X.columns.tolist()

In [ ]:
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=feature_names
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=feature_names,
    plot_type="bar"
)

In [ ]:
importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": best_rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

In [ ]:
plt.figure(figsize=(10,6))

plt.barh(
    importance["Feature"],
    importance["Importance"]
)

plt.gca().invert_yaxis()

plt.xlabel("Importance")

plt.title("Random Forest Feature Importance")

plt.show()

In [ ]:
import joblib

joblib.dump(
    best_rf,
    DATASET_DIR / "processed" / "best_resume_model.pkl"
)

print("Model saved successfully.")

In [ ]:
plt.figure(figsize=(7,7))

plt.scatter(y_test, pred, alpha=0.5)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--',
    linewidth=2
)

plt.xlabel("Actual Matching Score")
plt.ylabel("Predicted Matching Score")
plt.title("Actual vs Predicted Matching Score")

plt.grid(True)
plt.show()

In [ ]:
residuals = y_test - pred

plt.figure(figsize=(8,5))

plt.scatter(pred, residuals, alpha=0.5)

plt.axhline(0, color="red", linestyle="--")

plt.xlabel("Predicted Score")
plt.ylabel("Residual")

plt.title("Residual Analysis")

plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(residuals, bins=30)

plt.xlabel("Prediction Error")
plt.ylabel("Frequency")

plt.title("Distribution of Prediction Errors")

plt.show()

In [ ]:
resume_df["predicted_score"] = best_rf.predict(X_scaled)

In [ ]:
ranking = resume_df[
    [
        "job_position_name",
        "matched_score",
        "predicted_score"
    ]
].sort_values(
    by="predicted_score",
    ascending=False
)

ranking.head(20)

In [ ]:
ranking.to_csv(
    DATASET_DIR / "processed" / "final_candidate_ranking.csv",
    index=False
)

print("Ranking saved successfully.")

In [ ]:
resume_df.to_csv(
    DATASET_DIR / "processed" / "final_dataset_with_features.csv",
    index=False
)

print("Dataset saved successfully.")

In [ ]:
resume_df.to_csv(
    DATASET_DIR / "processed" / "final_dataset_with_features.csv",
    index=False
)

print("Dataset saved successfully.")

In [ ]:
metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Value": [
        mean_absolute_error(y_test, pred),
        mean_squared_error(y_test, pred) ** 0.5,
        r2_score(y_test, pred)
    ]
})

metrics

In [ ]:
metrics.to_csv(
    DATASET_DIR / "processed" / "evaluation_metrics.csv",
    index=False
)

In [ ]:
import joblib

joblib.dump(best_rf, "best_resume_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

In [ ]:
import joblib

joblib.dump(best_rf, "../models/best_resume_model.pkl")

joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

joblib.dump(scaler, "../models/scaler.pkl")

print("Models Saved Successfully")

In [ ]:
print(X.columns.tolist())